# GPS 

En este nuevo notebook, se centrara en como se esta construyendo este gps y su evolucion en el momento de realizar los calculos con los mejores rutas.

Así mismo esto dependera de manera amplia de forma concreta de todo lo que ingrese nuestro usuario, el tipo del mismo e incluso, para que se esta calculando la distancia e objetivo principal de esta. 

Con todo lo anterior se plantea como en la mayoria de los notebooks de este directorio, el uso de manhattan junto a otros algoritmos de búsqueda para mantener la mejor eficiencia en la ruta obtenida.

## Como se esta empleando **MANHATTAN**


Ahora bien manhattan se estara emplando de la siguiente forma

- Medicion de la cercania en los nodos
- Heuristica en el algoritmo de estrella (A*)
- Estimacion de los bloques urbanos

## Base matemática 

Una vez tenemos definido lo anterior podemos decir que con la mayoria de los requirimientos necesarios y solicitados para realizar de forma correcta se propone una función de *Costo Inteligente*

**Costo = distancia + trafico + riesgo + tiempo + restricciones**

En dónde se puede pasar de la siguiente forma    
    Costo=d+t+r+c+x, dentro de las cuáles cada letra esta definida de la siguiente manera.
    
    - D --> Es la distancia en Manhattan
    - t --> Es el tráfico
    - r --> riesgo
    - c --> Consumo 
    - x --> penalizaciones o restricciones 

Para ello se requiere de comprender o conocer un poco acerca de la base matematica con la que interactua un GPS, como lo son los algoritmos de busqueda o los de represntacion de rutas como lo es Dijsktra.

Asi como lo son el peso dinamico en los grafos, entre otros temas de importancia en la cosntruccion de algoritmos de busqueda. 

Saber distinguir entre las rutas que son optiminas no solo en la necesidad de ruta corta en kilometraje sino también en tiempo, seguridad de la ruta, tráfico, consumo de combustibles y más por eso se utuliza Manhattan sin embargo solo como heuristica. Agregando que la optimizacion de la busqueda de la ruta se tomara dependiendo del contexto. 

Redifiniendo la funcion anteriormente propuesta se plantea la siguiente como una función general
> Costo = d + b + r + g

En donde d es distancia, t es el tráfico o tiempo, r obviamente es el riesgo y por ultimo g lo definimos como giros de complejidad o otras restricciones

### Sub variante de la funcion de costo 

En esta sub variente se contempla otro tipo de restricciones de acuerdo al contexto de un camión de carga:

Es decir se agrega lo que es pendiente  y curvas de dificultad en maniobra

### Perfiles

Como se viene mencionando en puntos anteriores, ejempleficamos tres perfiles y como se comportará el GPS en función del perfil seleccionando con el único objetivo de ofrecer la mejor experencia y ruta posible al usuario. 

En cada uno de los perfiles que a continuacion se plantea se ejemplefica como se esta usando los coeficientes de la función de costo.

#### Usuario Común 

- d = medio 
- b = alto 
- r = bajo 
- g = bajo

Es la importancia de los coeficientes que rejiran la decision del modelo al calcular la ruta

#### Usuario Seguro

- d = bajo
- b = medio  
- r = alto 
- g = medio

Es la importancia de los coeficientes que rejiran la decision del modelo al calcular la ruta

#### Emergencia

- d = bajo 
- b = muy alto 
- r = bajo
- g = bajo

Es la importancia de los coeficientes que rejiran la decision del modelo al calcular la ruta

### Restricciones 

Como restricciones se contemplan que se penalicen de forma eficiente por ende se divide en dos clasificaciones, 1 Blandas y 2 Duras.

Por el lado de las blandas son:
- Trafico
- Riesgo
- Peaje 
- Calles incomodas

Para las restricciones duras son las siguientes:
- Cierre total 
- Altura insuficiente para camión 
- sentido contrario
- Acceso prohibido


Claramente estas resctricciones son a nivel general pero solo se aplicarán de acuerdo al contexto de la propia ruta

## Implementacion Matematica de Manhattan
Durante el siguiente apartado se hablara acerca de como se esta implementado manhattan como nuestra función de heuristica junto al algoritmo de A* (Búsqueda de estreslla). 

En esta parte se establece el 
- Recibir un grafo. 
- Recibir coordenadas de cada nodo 
- Usar distancia Manhattan como heuristica 
- Devuelva la mejor ruta 



In [4]:
import heapq

def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def reconstruir_ruta(padre, actual):
    ruta = [actual]

    while actual in padre:
        actual = padre[actual]
        ruta.append(actual)

    ruta.reverse()
    return ruta


def a_estrella(grafo, coords, inicio, meta):
    abiertos = []
    heapq.heappush(abiertos, (0, inicio))

    padre = {}

    g = {nodo: float("inf") for nodo in grafo}
    g[inicio] = 0

    f = {nodo: float("inf") for nodo in grafo}
    f[inicio] = manhattan(coords[inicio], coords[meta])

    cerrados = set()

    while abiertos:
        _, actual = heapq.heappop(abiertos)

        if actual == meta:
            return reconstruir_ruta(padre, actual), g[meta]

        if actual in cerrados:
            continue

        cerrados.add(actual)

        for vecino, costo in grafo[actual].items():
            nuevo_g = g[actual] + costo

            if nuevo_g < g[vecino]:
                padre[vecino] = actual
                g[vecino] = nuevo_g
                f[vecino] = nuevo_g + manhattan(coords[vecino], coords[meta])

                heapq.heappush(abiertos, (f[vecino], vecino))

    return None, float("inf")

In [2]:
# Construccion del grafo usando diccionarios junto a las coordenadas. 

grafo = {
    "A": {"B": 1, "D": 1},
    "B": {"A": 1, "C": 1, "E": 1},
    "C": {"B": 1, "F": 1},
    "D": {"A": 1, "E": 1},
    "E": {"B": 1, "D": 1, "F": 1},
    "F": {"C": 1, "E": 1}
}

coords = {
    "A": (0,0),
    "B": (1,0),
    "C": (2,0),
    "D": (0,1),
    "E": (1,1),
    "F": (2,1)
}

In [5]:
# Ejcucion del código.

ruta, costo = a_estrella(grafo, coords, "A", "F")

print (f'Ruat: {ruta}')
print (f'Costo: {costo}')

Ruat: ['A', 'B', 'C', 'F']
Costo: 3
